# Seismic Data Preprocessing & Instrument Response Removal
## From Raw MiniSEED Counts to Physical Ground Motion

**A comprehensive tutorial using ObsPy and SmartSolo IGU-16HR 3C data**

---

### Table of Contents

**Part 1 — Theory**
1. [The MiniSEED Format](#1-miniseed)
2. [What Does a Seismometer Actually Record?](#2-recording-chain)
3. [Instrument Response: Poles, Zeros & Transfer Functions](#3-instrument-response)
4. [Reading RESP Files Line by Line](#4-resp-files)
5. [Pole-Zero (SAC PZ) Files](#5-pz-files)
6. [StationXML / FDSN Inventory](#6-stationxml)
7. [Pre-processing Steps: Detrend, Demean, Taper, Filter](#7-preprocessing)
8. [Response Removal: Theory and Pitfalls](#8-response-removal-theory)

**Part 2 — Hands-On with GEONAT SmartSolo Data**
1. [Loading & Inspecting MiniSEED Data](#part2-1-loading)
2. [Comparing the SmartSolo 5 Hz and 2 Hz Geophones](#part2-2-instruments)
3. [Parsing and Understanding Your RESP Files](#part2-3-resp)
4. [The "fromMilliVolts" RESP — Why It Matters](#part2-4-millivolts)
5. [Full Preprocessing Pipeline](#part2-5-pipeline)
6. [Response Removal in Practice](#part2-6-removal)
7. [Polarity Issues with SmartSolo Nodes](#part2-7-polarity)
8. [Exercises — Verify Your Understanding](#part2-8-exercises)

---

In [ ]:
# Core imports used throughout this notebook
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
import obspy
from obspy import read, read_inventory, UTCDateTime
from obspy.core.inventory import read_inventory
from obspy.signal.invsim import corn_freq_2_paz, paz_to_freq_resp
import warnings
warnings.filterwarnings('ignore')

print(f"ObsPy version: {obspy.__version__}")

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100

---
# PART 1 — THEORY
---

<a id="1-miniseed"></a>
## 1. The MiniSEED Format

### What is MiniSEED?

**MiniSEED** is the subset of the SEED (Standard for the Exchange of Earthquake Data) format that contains only waveform data — no metadata. It is the *de facto* standard for distributing continuous seismic waveforms.

A MiniSEED file consists of **data records** (typically 512 or 4096 bytes each), each containing:

| Field | Description |
|-------|-------------|
| **Network** | 2-character network code (e.g., `SS` for SmartSolo) |
| **Station** | Up to 5-character station code (e.g., `00006`) |
| **Location** | 2-character location code (e.g., `SW`) |
| **Channel** | 3-character channel code (e.g., `DPZ`) |
| **Start time** | Timestamp of the first sample |
| **Sample rate** | Samples per second |
| **Data** | Compressed integer samples (Steim1 or Steim2 encoding) |

### Channel Naming Convention (SEED)

The 3-character channel code follows the pattern **`BLC`**:

- **B** (Band code): Indicates sample rate and response band
  - `D` = ≥250 sps, short period (our SmartSolo data at 250 sps)
  - `S` = 10–80 sps, short period
  - `H` = ≥80 sps, high broadband
  - `B` = 10–80 sps, broadband
  
- **L** (Source/instrument code):
  - `P` = Short-period geophone
  - `H` = High-gain seismometer
  
- **C** (Orientation code):
  - `Z` = Vertical
  - `N` = North-South
  - `E` = East-West

So **`DPZ`** = high-rate (≥250 sps) short-period geophone, vertical component.

### What are the values stored in MiniSEED?

**Raw digital counts** — integers that come directly from the analog-to-digital converter (ADC). These numbers have **no physical unit** until you apply the instrument response. A value of `15000` counts could represent wildly different ground motions depending on the instrument.

In [ ]:
# Quick demo: create a synthetic MiniSEED to show the format structure
from obspy import Trace, Stream

# Create a simple trace with synthetic data
tr = Trace(data=np.array([100, 200, 150, -50, -200, 300], dtype=np.int32))
tr.stats.network = 'SS'
tr.stats.station = '00006'
tr.stats.location = 'SW'
tr.stats.channel = 'DPZ'
tr.stats.sampling_rate = 250.0
tr.stats.starttime = UTCDateTime(2026, 9, 1)

print("=== Trace Stats ===")
print(tr.stats)
print(f"\nData type: {tr.data.dtype}")
print(f"Data values (raw counts): {tr.data}")
print(f"\nThese are dimensionless integers — NOT physical units!")

<a id="2-recording-chain"></a>
## 2. What Does a Seismometer Actually Record?

### The Recording Chain

Ground motion passes through a chain of transformations before becoming digital counts:

```
Ground Motion    →    Sensor (Geophone)    →     Digitizer (ADC)    →    Digital Counts
  [m/s]                   [V]                      [counts]              (stored in MiniSEED)
  velocity            analog voltage            integer samples
```

### Stage 1: The Geophone (Sensor)

A **geophone** is an electromagnetic sensor. A coil suspended on a spring moves relative to a magnet when the ground shakes. This relative motion induces a **voltage** (Faraday's law).

Key parameters:
- **Natural frequency** ($f_0$): The frequency at which the geophone resonates. Below $f_0$, sensitivity drops rapidly.
  - SmartSolo 5 Hz geophone: $f_0 = 5$ Hz  
  - SmartSolo 2 Hz geophone: $f_0 = 2$ Hz (extends sensitivity to lower frequencies)
- **Sensitivity** ($G$): Output voltage per unit ground velocity, in V/(m/s)
  - SmartSolo 5 Hz: ~76.6–76.8 V/(m/s) at reference frequency
  - SmartSolo 2 Hz: ~260 V/(m/s) at reference frequency
- **Damping** ($h$): Controls how quickly oscillations die out. Typically $h \approx 0.707$ (critical damping).

The geophone acts as a **velocity transducer**: its output voltage is proportional to ground velocity — but only above its natural frequency. Below $f_0$, the response rolls off.

### Stage 2: The Digitizer (ADC)

The analog voltage is sampled and converted to digital integers (counts). For SmartSolo nodes, the digitizer is integrated into the unit (it's an "all-in-one" system). The SmartSolo internal processing outputs data in **millivolts** (mV), not Volts.

### The Transfer Function

Mathematically, the full recording chain is a **convolution** in the time domain, or a **multiplication** in the frequency domain:

$$D(\omega) = G(\omega) \cdot I(\omega) \cdot A(\omega)$$

where:
- $D(\omega)$ = recorded digital output (counts or mV, depending on calibration)
- $G(\omega)$ = ground motion (what we want, in m/s or m/s²)
- $I(\omega)$ = instrument response (transfer function of sensor)
- $A(\omega)$ = digitizer response

**Response removal** = dividing out $I(\omega)$ and $A(\omega)$ to recover $G(\omega)$:

$$G(\omega) = \frac{D(\omega)}{I(\omega) \cdot A(\omega)}$$

<a id="3-instrument-response"></a>
## 3. Instrument Response: Poles, Zeros & Transfer Functions

### The Laplace Domain

The instrument response is described as a **rational function** in the Laplace domain ($s = j\omega$, where $\omega = 2\pi f$):

$$H(s) = A_0 \cdot G \cdot \frac{\prod_{k=1}^{M}(s - z_k)}{\prod_{k=1}^{N}(s - p_k)}$$

where:
- $A_0$ = **normalization factor** (makes $|H(s)|=1$ at the normalization frequency when multiplied by the gain)
- $G$ = **stage gain** (sensitivity in V/(m/s) or mV/(m/s))
- $z_k$ = **zeros** of the transfer function (complex numbers)
- $p_k$ = **poles** of the transfer function (complex numbers)

### What are Poles and Zeros?

**Zeros** ($z_k$): Frequencies where the response goes to **zero** (output = 0 regardless of input).
- A zero at the origin ($z = 0$) means the instrument doesn't respond to DC (static offset).
- Two zeros at origin → the instrument is a **velocity sensor** (response to displacement is zero at 0 Hz).

**Poles** ($p_k$): Frequencies where the response goes to **infinity** (in theory). They define the resonance characteristics.
- Poles always come in **conjugate pairs** for physical systems: $p = \sigma \pm j\omega_d$
- The natural frequency: $\omega_0 = \sqrt{\sigma^2 + \omega_d^2}$
- The damping ratio: $h = -\sigma / \omega_0$

### SmartSolo 5 Hz Geophone: Poles & Zeros

```
Zeros: (0, 0), (0, 0)           ← Two zeros at origin → velocity sensor
Poles: (-22.211 ± 22.218j)      ← Conjugate pair → resonance at ~5 Hz
```

Let's verify: $\omega_0 = \sqrt{22.211^2 + 22.218^2} = \sqrt{987.73}$ ≈ 31.43 rad/s → $f_0 = 31.43/(2\pi)$ ≈ **5.0 Hz** ✓

### SmartSolo 2 Hz Geophone: Poles & Zeros

```
Zeros: (0, 0), (0, 0)
Poles: (-8.8844 ± 8.8871j)      ← resonance at ~2 Hz
```

$\omega_0 = \sqrt{8.8844^2 + 8.8871^2}$ ≈ 12.566 rad/s → $f_0 = 12.566/(2\pi)$ ≈ **2.0 Hz** ✓

In [ ]:
# Let's visualize the poles and zeros and compute the transfer function

# SmartSolo 5 Hz geophone poles & zeros
zeros_5hz = [0+0j, 0+0j]
poles_5hz = [-22.211059+22.217768j, -22.211059-22.217768j]
gain_5hz = 76.7  # V/(m/s) — from PZ file CONSTANT

# SmartSolo 2 Hz geophone poles & zeros
zeros_2hz = [0+0j, 0+0j]
poles_2hz = [-8.8844236+8.8871072j, -8.8844236-8.8871072j]
gain_2hz = 260.0  # V/(m/s)

# Verify natural frequencies
for name, poles in [('5 Hz', poles_5hz), ('2 Hz', poles_2hz)]:
    p = poles[0]
    omega0 = np.sqrt(p.real**2 + p.imag**2)
    f0 = omega0 / (2 * np.pi)
    damping = -p.real / omega0
    print(f"SmartSolo {name}: f0 = {f0:.2f} Hz, damping = {damping:.4f}, omega0 = {omega0:.3f} rad/s")

In [ ]:
# Compute and plot the amplitude response of both geophones

def compute_response(freqs, zeros, poles, gain, a0=1.0):
    """Compute the complex frequency response from poles and zeros.
    
    H(s) = A0 * gain * prod(s - z_k) / prod(s - p_k)
    where s = j*omega = j*2*pi*f
    """
    omega = 2 * np.pi * freqs
    s = 1j * omega
    
    numerator = np.ones(len(freqs), dtype=complex)
    for z in zeros:
        numerator *= (s - z)
    
    denominator = np.ones(len(freqs), dtype=complex)
    for p in poles:
        denominator *= (s - p)
    
    return a0 * gain * numerator / denominator

freqs = np.logspace(-1, 2.3, 1000)  # 0.1 to ~200 Hz

resp_5hz = compute_response(freqs, zeros_5hz, poles_5hz, gain_5hz)
resp_2hz = compute_response(freqs, zeros_2hz, poles_2hz, gain_2hz)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Amplitude response
ax1.loglog(freqs, np.abs(resp_5hz), 'b-', linewidth=2, label='SmartSolo 5 Hz (IGU-16HR 3C)')
ax1.loglog(freqs, np.abs(resp_2hz), 'r-', linewidth=2, label='SmartSolo 2 Hz (IGU-16HR 3C 2Hz)')
ax1.axvline(5, color='b', linestyle='--', alpha=0.5, label='f₀ = 5 Hz')
ax1.axvline(2, color='r', linestyle='--', alpha=0.5, label='f₀ = 2 Hz')
ax1.set_ylabel('Amplitude [V/(m/s)]')
ax1.set_title('SmartSolo Geophone Amplitude Response')
ax1.legend()
ax1.grid(True, which='both', alpha=0.3)
ax1.set_ylim(1e-2, 1e4)

# Phase response
ax2.semilogx(freqs, np.degrees(np.angle(resp_5hz)), 'b-', linewidth=2, label='5 Hz')
ax2.semilogx(freqs, np.degrees(np.angle(resp_2hz)), 'r-', linewidth=2, label='2 Hz')
ax2.axvline(5, color='b', linestyle='--', alpha=0.5)
ax2.axvline(2, color='r', linestyle='--', alpha=0.5)
ax2.set_xlabel('Frequency [Hz]')
ax2.set_ylabel('Phase [degrees]')
ax2.set_title('Phase Response')
ax2.legend()
ax2.grid(True, which='both', alpha=0.3)
ax2.set_ylim(-200, 200)

plt.tight_layout()
plt.show()

print("\n→ Key observation: Below the natural frequency, the response drops off at 12 dB/octave")
print("  (because of the two zeros at the origin). The 2 Hz geophone extends usable response")
print("  to lower frequencies than the 5 Hz, but the 5 Hz has a flatter passband above 5 Hz.")

In [ ]:
# Pole-Zero plot in the complex s-plane
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

for ax, name, zeros, poles, color in [
    (ax1, 'SmartSolo 5 Hz', zeros_5hz, poles_5hz, 'blue'),
    (ax2, 'SmartSolo 2 Hz', zeros_2hz, poles_2hz, 'red')]:
    
    # Plot zeros as 'o' and poles as 'x'
    for z in zeros:
        ax.plot(z.real, z.imag, 'o', markersize=12, color=color, 
                markerfacecolor='none', markeredgewidth=2)
    for p in poles:
        ax.plot(p.real, p.imag, 'x', markersize=12, color=color, 
                markeredgewidth=2)
    
    ax.axhline(0, color='k', linewidth=0.5)
    ax.axvline(0, color='k', linewidth=0.5)
    ax.set_xlabel('Real [rad/s]')
    ax.set_ylabel('Imaginary [rad/s]')
    ax.set_title(f'{name} — Pole-Zero Map\n(○ = zeros, × = poles)')
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

print("The zeros at the origin mean the sensor has no DC response (it measures velocity, not displacement).")
print("The conjugate pole pair defines the natural frequency and damping of the geophone.")

<a id="4-resp-files"></a>
## 4. Reading RESP Files Line by Line

A **RESP file** (SEED response file) is a human-readable ASCII description of the full instrument response, organized in **stages**. Let's dissect a real one from your GEONAT project.

### Anatomy of the RESP File: `DT-SOLO.5Hz.RESP`

```
#  << RESP_VER  RESP.XX.NS680..SPZ.DTSOLO.5.1850.43000.76_6  2020.043  NEW >>
```
↑ Header: version identifier, station `NS680`, network `XX`, channel `SPZ`.

---

#### Header Block (B050/B052)
```
B050F03     Station:     NS680
B050F16     Network:     XX
B052F03     Location:    ??
B052F04     Channel:     SPZ
B052F22     Start date:  2006,001,00:00:00.0000
B052F23     End date:    No Ending Time
```
- `B050`, `B052` refer to SEED **blockette** numbers (data structure types in the SEED standard)
- `F03`, `F16` are **field** numbers within each blockette
- `XX` = generic/unknown network code
- `NS680` = placeholder station name for the DT-SOLO 5 Hz instrument
- `SPZ` = Short-Period, vertical component (note: `S` not `D` because this RESP was written for a nominal 10-80 sps response specification, though your data records at 250 sps as `DPZ`)

---

#### Stage 1 — Poles & Zeros (B053): The Geophone Response
```
B053F03     Transfer function type:           A [Laplace Transform (Rad/sec)]
B053F04     Stage sequence number:            1
B053F05     Response in units lookup:         M/S - Velocity in Meters Per Second
B053F06     Response out units lookup:        V - Volts
B053F07     A0 normalization factor:          1.000000E+00
B053F08     Normalization frequency:          5.000000E+01
B053F09     Number of zeroes:                 2
B053F14     Number of poles:                  2
```

Key fields:
- **Transfer function type `A`**: Laplace (analog), in radians/second
- **Input: M/S** → Output: **V** (Volts): This stage converts ground velocity to voltage
- **A0 = 1.0**: The normalization factor is 1 (meaning the gain is already normalized)
- **Normalization frequency = 50 Hz**: The frequency at which A0 is defined

Then the zeros and poles:
```
B053F10-13    0   0.000000e+00   0.000000e+00    ← Zero 0: (0 + 0j)
B053F10-13    1   0.000000e+00   0.000000e+00    ← Zero 1: (0 + 0j)
B053F15-18    0  -2.221110e+01  -2.221780e+01    ← Pole 0: (-22.211 - 22.218j)
B053F15-18    1  -2.221110e+01   2.221780e+01    ← Pole 1: (-22.211 + 22.218j)
```

#### Stage 1 — Gain (B058)
```
B058F03     Stage sequence number:            1
B058F04     Gain:                             7.660000e+01    ← 76.6 V/(m/s)
B058F05     Frequency of gain:                5.000000e+01 HZ
```

This is the **stage gain** at 50 Hz: the geophone produces 76.6 Volts per (m/s) of ground velocity at 50 Hz.

---

#### Stage 2 — Digital Coefficients (B054): The Digitizer
```
B054F03     Transfer function type:           D        ← Digital
B054F04     Stage sequence number:            2
B054F05     Response in units lookup:         V - Volts
B054F06     Response out units lookup:        COUNTS - Digital Counts
B054F07     Number of numerators:             1
B054F08-09  0   1.000000e+00                  ← coefficient = 1.0 (unity)
```

**Important**: The digitizer coefficient is 1.0 with gain 1.0 in stage 2. This means the RESP file encodes the *entire* system sensitivity in stage 1.

#### Stage 2 — Decimation (B057)
```
B057F04     Input sample rate:                4.000000e+01    ← 40 sps (nominal)
B057F05     Decimation factor:                1
```

#### Stage 0 — Overall Sensitivity (B058)
```
B058F03     Stage sequence number:            0
B058F04     Sensitivity:                      7.660000e+01    ← 76.6 overall
B058F05     Frequency of sensitivity:         5.000000e+01 HZ
```

**Stage 0 is always the overall sensitivity** = product of all stage gains = 76.6 × 1.0 = 76.6 V/(m/s).

### The crucial difference: `DT-SOLO.5Hz.RESP` vs `RESP...fromMilliVolts`

| Property | DT-SOLO.5Hz.RESP | fromMilliVolts RESP |
|----------|-----------------|--------------------|
| Stage 1 Gain | **76.6** (V/(m/s)) | **76,600** (mV/(m/s)) |
| Overall Sensitivity | **76.6** | **76,600** |
| Output units | **Volts** | **milliVolts** |

The "fromMilliVolts" version has the gain multiplied by **1000** because SmartSolo internally outputs data in **millivolts**, not volts. If your raw data is in mV (as the SmartSolo FDSN XML confirms: `<OutputUnits><Name>mV</Name>`), then you **must** use the `fromMilliVolts` RESP to get correct physical units after response removal.

<a id="5-pz-files"></a>
## 5. Pole-Zero (SAC PZ) Files

A **SAC Pole-Zero** file is a simpler format that contains the same poles-and-zeros information as a RESP file, but condensed:

### `smartsolo.5Hz.pz`:
```
ZEROS 2
0 0
0 0
POLES 2
-22.211059    22.217768
-22.211059   -22.217768
CONSTANT 76.7
```

The **CONSTANT** = $A_0 \times G$ = normalization factor × sensitivity = 1.0 × 76.7 = 76.7 V/(m/s).

### `smartsolo.2Hz.pz`:
```
ZEROS 2
0 0
0 0
POLES 2
-8.8844236    8.8871072
-8.8844236   -8.8871072
CONSTANT 260
```

The 2 Hz geophone has higher sensitivity (260 V/(m/s)) and lower natural frequency poles.

### `IGU-BD3C-5_PZ_Pole.pz` — A More Complex Response:
```
ZEROS 4
1.4164e4 0.
-7.162e3 0.
0. 0.
0. 0.
POLES 3
-1.7204e3 0.
-1.2 9.0e-1
-1.2 -9.0e-1
CONSTANT -3.554090e-03
```

This file has **4 zeros** and **3 poles** — it includes **additional stages** of the electronics (anti-alias filters, etc.) folded into a single poles-and-zeros representation. The **negative CONSTANT** indicates a **polarity reversal** in the instrument (more on this in Part 2, Section 7).

Note: PZ files combine all stages into one transfer function. RESP files keep stages separate, which provides more detail but is more verbose.

<a id="6-stationxml"></a>
## 6. StationXML / FDSN Inventory

**StationXML** (also called FDSN StationXML) is the modern XML replacement for SEED metadata. It contains the same information as RESP files but in a structured, machine-readable format.

SmartSolo exports FDSN StationXML files alongside the MiniSEED data. These contain:
- Station coordinates (latitude, longitude, elevation)
- Channel orientation (azimuth, dip)
- Full response information (poles, zeros, gains)

Example from your GEONAT FDSN XML:
```xml
<InstrumentSensitivity>
  <Value>7.68e4</Value>               ← 76,800 mV/(m/s)
  <Frequency>50</Frequency>            ← at 50 Hz
  <InputUnits><Name>m/s</Name></InputUnits>
  <OutputUnits><Name>mV</Name></OutputUnits>  ← millivolts!
</InstrumentSensitivity>
```

**This confirms**: SmartSolo raw output is in **millivolts**, and the overall sensitivity is ~76,800 mV/(m/s).

In ObsPy, you load this as an `Inventory` object:
```python
inv = read_inventory('FDSN_Information.xml')
```

<a id="7-preprocessing"></a>
## 7. Pre-processing Steps: Detrend, Demean, Taper, Filter

Before removing the instrument response, you **must** pre-process the raw data. Skipping these steps leads to artifacts — sometimes catastrophically bad ones.

### 7.1 Why Pre-process?

Response removal involves **spectral division** — dividing the data spectrum by the instrument response. Any low-frequency noise (trends, offsets) gets amplified enormously when divided by the vanishing response at low frequencies. Pre-processing mitigates this.

### 7.2 Demean

Remove the mean value from the trace:
$$x'(t) = x(t) - \bar{x}$$

**Why?** A non-zero mean appears as a spike at 0 Hz in the spectrum. When you divide by the instrument response (which goes to zero at DC due to the zeros at the origin), this spike gets amplified to infinity.

```python
tr.detrend('demean')
```

### 7.3 Detrend

Remove a linear trend (or higher-order polynomial):
$$x'(t) = x(t) - (at + b)$$

**Why?** Instrumental drift, temperature effects, or tilt cause slow trends that concentrate energy at very low frequencies. A linear detrend removes the dominant trend.

```python
tr.detrend('linear')      # Remove best-fit line
tr.detrend('polynomial', order=3)  # For more complex drift
```

**Types of detrend in ObsPy:**
- `'demean'`: subtract the mean
- `'linear'`: subtract best-fit line (includes demean)
- `'polynomial'`: subtract best-fit polynomial of given order
- `'spline'`: subtract a spline fit (for complex drift patterns)
- `'simple'`: subtract line through first and last sample

### 7.4 Taper

Apply a **window function** that smoothly tapers the edges of the trace to zero:

**Why?** The FFT assumes the signal is periodic. If the beginning and end of your trace don't match (they never do), this discontinuity introduces **spectral leakage** — artificial broadband energy. Tapering eliminates the discontinuity.

```python
tr.taper(max_percentage=0.05, type='cosine')  # 5% Tukey/cosine taper on each side
```

Common taper types:
- **Cosine (Tukey)**: Most common. `max_percentage=0.05` tapers the first/last 5%.
- **Hann**: Stronger taper but loses more data at the edges.

**Rule of thumb**: Use 5% cosine taper for routine processing.

### 7.5 Bandpass Filter

Restrict the frequency content to a range of interest:

```python
tr.filter('bandpass', freqmin=0.5, freqmax=20.0, corners=4, zerophase=True)
```

**Why?** 
- High frequencies contain mostly noise above the Nyquist limit of interest
- Low frequencies below the geophone's natural frequency have very low SNR
- Filtering *after* response removal can also stabilize the result

**Note**: You can either:
1. Filter before response removal (to limit the bandwidth)
2. Use the `pre_filt` parameter during response removal (recommended — see next section)

In [ ]:
# Demonstrate preprocessing effects on synthetic data

np.random.seed(42)
npts = 10000
dt = 1.0 / 250.0  # 250 sps
t = np.arange(npts) * dt

# Synthetic seismic trace: signal + trend + offset + noise
signal = 500 * np.sin(2 * np.pi * 5 * t) * np.exp(-0.5 * (t - 20)**2 / 4)  # 5 Hz burst at t=20s
trend = 50 * t + 2000  # Linear drift + DC offset
noise = 30 * np.random.randn(npts)
raw = signal + trend + noise

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

# Raw
axes[0].plot(t, raw, 'k', linewidth=0.5)
axes[0].set_title('(a) Raw data: signal + linear trend + DC offset + noise')
axes[0].set_ylabel('Counts')

# After demean
demeaned = raw - np.mean(raw)
axes[1].plot(t, demeaned, 'b', linewidth=0.5)
axes[1].set_title('(b) After demean: DC offset removed, but trend remains')
axes[1].set_ylabel('Counts')

# After detrend (linear)
from scipy.signal import detrend
detrended = detrend(raw, type='linear')
axes[2].plot(t, detrended, 'g', linewidth=0.5)
axes[2].set_title('(c) After linear detrend: trend AND mean removed')
axes[2].set_ylabel('Counts')

# After detrend + taper
taper_width = int(0.05 * npts)
taper = np.ones(npts)
taper[:taper_width] = 0.5 * (1 - np.cos(np.pi * np.arange(taper_width) / taper_width))
taper[-taper_width:] = 0.5 * (1 - np.cos(np.pi * np.arange(taper_width, 0, -1) / taper_width))
tapered = detrended * taper
axes[3].plot(t, tapered, 'r', linewidth=0.5)
axes[3].plot(t, taper * detrended.max(), 'k--', linewidth=1, alpha=0.3, label='Taper window')
axes[3].set_title('(d) After detrend + 5% cosine taper: edges smoothly go to zero')
axes[3].set_ylabel('Counts')
axes[3].set_xlabel('Time [s]')
axes[3].legend()

plt.tight_layout()
plt.show()

<a id="8-response-removal-theory"></a>
## 8. Response Removal: Theory and Pitfalls

### The Process

Response removal is a **spectral division** operation:

1. **FFT** the time series to get the spectrum $D(\omega)$
2. **Compute** the instrument response $I(\omega)$ from poles/zeros/gain
3. **Divide**: $G(\omega) = D(\omega) / I(\omega)$
4. **Inverse FFT** to get the corrected time series $g(t)$

### The Water Level Problem

The instrument response $I(\omega)$ goes to **zero** at some frequencies (e.g., at DC for a velocity sensor). Dividing by zero (or near-zero) amplifies noise to infinity.

**Solution: `pre_filt`** — a bandpass filter applied *in the frequency domain* before division:

```python
pre_filt = [f1, f2, f3, f4]  # Four corner frequencies in Hz
```

This creates a **cosine taper** in the frequency domain:
```
Amplitude
1.0 |          _______________
    |         /               \
    |        /                 \
    |       /                   \
0.0 |______/                     \______
    f1    f2                   f3    f4    Frequency
```

- Below `f1` and above `f4`: response is zeroed out (no correction)
- Between `f1`–`f2` and `f3`–`f4`: cosine ramp (smooth transition)
- Between `f2` and `f3`: full correction applied

### Choosing `pre_filt`

Guidelines for SmartSolo geophones:

| Geophone | Suggested `pre_filt` | Rationale |
|----------|---------------------|-----------|
| 5 Hz | `[0.5, 1.0, 100, 120]` | Start above microseism band, well below Nyquist |
| 2 Hz | `[0.2, 0.5, 100, 120]` | Can go lower thanks to lower natural frequency |

- `f1`, `f2`: Should be at or below the geophone natural frequency. Going too low amplifies noise.
- `f3`, `f4`: Should be below Nyquist (125 Hz for 250 sps) to avoid aliasing artifacts.

### Output Options

After response removal, you choose the output units:

| `output` parameter | Physical units | When to use |
|-------------------|---------------|-------------|
| `'VEL'` | m/s (velocity) | Standard for most seismology |
| `'DISP'` | m (displacement) | For surface waves, geodesy |
| `'ACC'` | m/s² (acceleration) | For strong motion, engineering |

```python
# In ObsPy:
tr.remove_response(inventory=inv, output='VEL', pre_filt=pre_filt)
```

### Common Pitfalls

1. **Forgetting to detrend/demean** → Massive low-frequency artifacts
2. **pre_filt too wide** → Amplifying noise at frequencies where SNR is low
3. **Wrong RESP/sensitivity** → Output amplitude off by orders of magnitude
4. **Using Volts RESP when data is in millivolts** → Output off by factor of 1000
5. **Not accounting for polarity reversal** → Waveforms are upside down

In [ ]:
# Demonstrate the water-level / pre_filt concept

freqs_demo = np.linspace(0, 125, 10000)
freqs_demo[0] = 0.001  # avoid division by zero

# Compute the 5 Hz geophone response amplitude
resp = compute_response(freqs_demo, zeros_5hz, poles_5hz, gain_5hz)
resp_amp = np.abs(resp)

# pre_filt: [f1, f2, f3, f4]
f1, f2, f3, f4 = 0.5, 1.0, 100, 120

# Build the pre_filt taper
pre_filt_taper = np.zeros(len(freqs_demo))
for i, f in enumerate(freqs_demo):
    if f < f1:
        pre_filt_taper[i] = 0
    elif f < f2:
        pre_filt_taper[i] = 0.5 * (1 - np.cos(np.pi * (f - f1) / (f2 - f1)))
    elif f < f3:
        pre_filt_taper[i] = 1.0
    elif f < f4:
        pre_filt_taper[i] = 0.5 * (1 + np.cos(np.pi * (f - f3) / (f4 - f3)))
    else:
        pre_filt_taper[i] = 0

# What happens when we divide: 1/response
correction_no_filt = 1.0 / resp_amp
correction_with_filt = pre_filt_taper / resp_amp

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

ax1.semilogy(freqs_demo, resp_amp, 'b-', linewidth=2, label='Instrument response |H(f)|')
ax1.semilogy(freqs_demo, pre_filt_taper * resp_amp.max(), 'r--', linewidth=2, label='pre_filt window (scaled)')
ax1.axvline(5, color='gray', linestyle=':', alpha=0.5, label='Natural freq (5 Hz)')
ax1.set_ylabel('Amplitude')
ax1.set_title('Instrument Response and pre_filt Window')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, 130)

ax2.semilogy(freqs_demo, correction_no_filt, 'k-', linewidth=1, alpha=0.5, label='1/|H(f)| — no pre_filt (DANGEROUS!)')
ax2.semilogy(freqs_demo, np.maximum(correction_with_filt, 1e-10), 'r-', linewidth=2, label='pre_filt / |H(f)| — safe correction')
ax2.set_xlabel('Frequency [Hz]')
ax2.set_ylabel('Correction factor')
ax2.set_title('Correction Factor With and Without pre_filt')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, 130)
ax2.set_ylim(1e-5, 1e5)

plt.tight_layout()
plt.show()

print("Without pre_filt: the correction factor explodes at low frequencies → amplifies noise.")
print("With pre_filt: the correction is smoothly tapered to zero outside the useful bandwidth.")

---
# PART 2 — HANDS-ON WITH GEONAT SMARTSOLO DATA
---

**Data location on the server (ssh mibam):**
```
/srv/beegfs/scratch/users/h/henrymi/project/GEONAT/miniseed_2Hz/   ← SmartSolo IGU-16HR 3C 2Hz (250 sps)
/srv/beegfs/scratch/users/h/henrymi/project/GEONAT/miniseed_5Hz/   ← SmartSolo IGU-16HR 3C 5Hz (250 sps)
/srv/beegfs/scratch/users/h/henrymi/project/GEONAT/RESP/           ← Response files
```

**Data summary:**
- Network: `SS`, Location: `SW`
- Channels: `DPZ`, `DPN`, `DPE` (vertical, north, east)
- 2 Hz stations: 00006, 00011, 00017, 00023, 00029, ... (1720 files)
- 5 Hz stations: 06384, 06391, 06397, 06398, 06400, ... (21055 files)
- Recording period: 2026-08-29 to 2026-09-16

In [ ]:
# === CONFIGURATION — EDIT THESE PATHS ===
# These paths are for running on the server (ssh mibam)

BASE_DIR = '/srv/beegfs/scratch/users/h/henrymi/project/GEONAT'

MSEED_2HZ_DIR = f'{BASE_DIR}/miniseed_2Hz'
MSEED_5HZ_DIR = f'{BASE_DIR}/miniseed_5Hz'
RESP_DIR = f'{BASE_DIR}/RESP'
RESP_MILLIVOLTS = f'{BASE_DIR}/RESP.XX.NS680..SPZ.DTSOLO.5.1850.43000.76_6_fromMilliVolts'
FDSN_DIR_2HZ = f'{MSEED_2HZ_DIR}/FDSN Information'
FDSN_DIR_5HZ = f'{MSEED_5HZ_DIR}/FDSN Information'

# We'll work with one example station from each instrument type
STATION_2HZ = '00006'  # 2 Hz geophone
STATION_5HZ = '06384'  # 5 Hz geophone

<a id="part2-1-loading"></a>
## Part 2.1 — Loading & Inspecting MiniSEED Data

In [ ]:
import glob
import os

# Find files for our example stations
files_2hz_z = sorted(glob.glob(f'{MSEED_2HZ_DIR}/SS.{STATION_2HZ}.SW.DPZ.*.miniseed'))
files_5hz_z = sorted(glob.glob(f'{MSEED_5HZ_DIR}/SS.{STATION_5HZ}.SW.DPZ.*.miniseed'))

print(f"Station {STATION_2HZ} (2 Hz), vertical component:")
for f in files_2hz_z[:3]:
    print(f"  {os.path.basename(f)}")
print(f"  ... ({len(files_2hz_z)} files total)")

print(f"\nStation {STATION_5HZ} (5 Hz), vertical component:")
for f in files_5hz_z[:3]:
    print(f"  {os.path.basename(f)}")
print(f"  ... ({len(files_5hz_z)} files total)")

In [ ]:
# Load one day of data from each instrument
# Pick the second file (first full day, typically 2026-08-30)

st_2hz = read(files_2hz_z[1])
st_5hz = read(files_5hz_z[1])

print("=== 2 Hz Station ===")
print(st_2hz)
print(st_2hz[0].stats)
print(f"Data range: {st_2hz[0].data.min()} to {st_2hz[0].data.max()} counts")
print(f"Data type: {st_2hz[0].data.dtype}")

print("\n=== 5 Hz Station ===")
print(st_5hz)
print(st_5hz[0].stats)
print(f"Data range: {st_5hz[0].data.min()} to {st_5hz[0].data.max()} counts")
print(f"Data type: {st_5hz[0].data.dtype}")

In [ ]:
# Plot raw data — these are raw digital counts (or millivolts, depending on SmartSolo config)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=False)

# Plot first 60 seconds
tr_2hz = st_2hz[0].copy()
tr_5hz = st_5hz[0].copy()

t_start = tr_2hz.stats.starttime
tr_2hz_clip = tr_2hz.slice(t_start, t_start + 60)
t_start_5 = tr_5hz.stats.starttime
tr_5hz_clip = tr_5hz.slice(t_start_5, t_start_5 + 60)

ax1.plot(tr_2hz_clip.times(), tr_2hz_clip.data, 'b-', linewidth=0.3)
ax1.set_title(f'2 Hz Station {STATION_2HZ} — Raw Counts (first 60s)')
ax1.set_ylabel('Counts / mV')
ax1.grid(True, alpha=0.3)

ax2.plot(tr_5hz_clip.times(), tr_5hz_clip.data, 'r-', linewidth=0.3)
ax2.set_title(f'5 Hz Station {STATION_5HZ} — Raw Counts (first 60s)')
ax2.set_ylabel('Counts / mV')
ax2.set_xlabel('Time [s]')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("These are raw values from the MiniSEED files.")
print("SmartSolo outputs in millivolts (mV), not raw ADC counts.")
print("To convert to physical ground motion, we need to remove the instrument response.")

In [ ]:
# Let's also look at the frequency content of the raw data

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for ax, tr, name, color in [(ax1, tr_2hz, f'2 Hz ({STATION_2HZ})', 'blue'), 
                             (ax2, tr_5hz, f'5 Hz ({STATION_5HZ})', 'red')]:
    # Use a copy and trim to manageable size
    tr_copy = tr.copy()
    tr_copy.detrend('linear')
    tr_copy.taper(max_percentage=0.01)
    
    nfft = len(tr_copy.data)
    freqs_fft = np.fft.rfftfreq(nfft, d=tr_copy.stats.delta)
    spectrum = np.abs(np.fft.rfft(tr_copy.data))
    
    ax.loglog(freqs_fft[1:], spectrum[1:], color=color, linewidth=0.3, alpha=0.7)
    ax.set_xlabel('Frequency [Hz]')
    ax.set_ylabel('Spectral Amplitude')
    ax.set_title(f'{name} — Raw Amplitude Spectrum')
    ax.grid(True, which='both', alpha=0.3)
    ax.set_xlim(0.01, 125)

plt.tight_layout()
plt.show()

<a id="part2-2-instruments"></a>
## Part 2.2 — Comparing the SmartSolo 5 Hz and 2 Hz Geophones

### SmartSolo IGU-16HR 3C (5 Hz)
- **Geophone**: 5 Hz natural frequency, 3-component
- **Sensitivity**: ~76.6–76.8 V/(m/s) (or ~76,600–76,800 mV/(m/s))
- **Sample rate**: 250 sps (stored as `DP*` channels)
- **Usable bandwidth**: ~1–100 Hz (with response correction)
- **Best for**: Local/regional seismology, ambient noise, active source surveys

### SmartSolo IGU-16HR 3C 2Hz
- **Geophone**: 2 Hz natural frequency, 3-component (extended low-frequency response)
- **Sensitivity**: ~260 V/(m/s)
- **Sample rate**: 250 sps
- **Usable bandwidth**: ~0.5–100 Hz (with response correction)
- **Best for**: Teleseismic events, surface wave analysis, deeper ambient noise tomography

The 2 Hz version costs more but provides better low-frequency sensitivity — critical for studying teleseismic events or long-period surface waves.

<a id="part2-3-resp"></a>
## Part 2.3 — Parsing and Understanding Your RESP Files

Let's load the RESP files with ObsPy and verify we understand them correctly.

In [ ]:
# Read the RESP files using ObsPy's evalresp-based reader
# ObsPy can read RESP files directly into an Inventory

from obspy.core.inventory.response import Response

# Read the standard 5 Hz RESP
resp_5hz_file = f'{RESP_DIR}/DT-SOLO.5Hz.RESP'
resp_mv_file = RESP_MILLIVOLTS

# ObsPy's read_inventory can parse RESP files
inv_5hz = read_inventory(resp_5hz_file, format='RESP')
inv_mv = read_inventory(resp_mv_file, format='RESP')

print("=== Standard 5 Hz RESP ===")
print(inv_5hz)
print()

print("=== fromMilliVolts RESP ===")
print(inv_mv)

In [ ]:
# Extract and compare the response details

for inv, label in [(inv_5hz, 'Standard RESP'), (inv_mv, 'fromMilliVolts RESP')]:
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    
    channel = inv[0][0][0]  # network → station → channel
    response = channel.response
    
    print(f"Station: {inv[0][0].code}")
    print(f"Channel: {channel.code}")
    print(f"\nOverall sensitivity: {response.instrument_sensitivity.value}")
    print(f"Sensitivity freq:    {response.instrument_sensitivity.frequency} Hz")
    print(f"Input units:         {response.instrument_sensitivity.input_units}")
    print(f"Output units:        {response.instrument_sensitivity.output_units}")
    
    for stage in response.response_stages:
        print(f"\n  Stage {stage.stage_sequence_number}:")
        print(f"    Gain: {stage.stage_gain}")
        print(f"    Gain freq: {stage.stage_gain_frequency} Hz")
        if hasattr(stage, 'pz_transfer_function_type'):
            print(f"    Transfer type: {stage.pz_transfer_function_type}")
            print(f"    Normalization factor: {stage.normalization_factor}")
            print(f"    Zeros: {stage.zeros}")
            print(f"    Poles: {stage.poles}")
            print(f"    Input units: {stage.input_units}")
            print(f"    Output units: {stage.output_units}")

In [ ]:
# Plot both response curves to see the difference

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

# Use ObsPy's built-in response plotting
for inv, label, color in [(inv_5hz, 'Standard (V)', 'blue'), 
                           (inv_mv, 'fromMilliVolts (mV)', 'red')]:
    response = inv[0][0][0].response
    resp_out, freqs_out = response.get_evalresp_response(
        t_samp=1.0/250.0, nfft=2**16, output='VEL'
    )
    
    ax1.loglog(freqs_out, np.abs(resp_out), color=color, linewidth=2, label=label)
    ax2.semilogx(freqs_out, np.degrees(np.angle(resp_out)), color=color, linewidth=2, label=label)

ax1.set_ylabel('Amplitude')
ax1.set_title('Amplitude Response: Standard vs fromMilliVolts RESP')
ax1.legend()
ax1.grid(True, which='both', alpha=0.3)
ax1.set_xlim(0.1, 125)

ax2.set_xlabel('Frequency [Hz]')
ax2.set_ylabel('Phase [degrees]')
ax2.set_title('Phase Response (identical shape — only amplitude differs by ×1000)')
ax2.legend()
ax2.grid(True, which='both', alpha=0.3)
ax2.set_xlim(0.1, 125)

plt.tight_layout()
plt.show()

print("The shapes are IDENTICAL — the only difference is the amplitude scale (×1000).")
print("Use the fromMilliVolts version for SmartSolo data since the output is in mV.")

<a id="part2-4-millivolts"></a>
## Part 2.4 — The "fromMilliVolts" RESP — Why It Matters

### The Problem

SmartSolo instruments are unusual: their internal digitizer outputs data in **millivolts (mV)**, not raw ADC counts or Volts. This is confirmed by the FDSN StationXML:

```xml
<OutputUnits><Name>mV</Name><Description>millivolt</Description></OutputUnits>
```

### What Goes Wrong with the Standard RESP

The standard `DT-SOLO.5Hz.RESP` file has:
- Stage 1 gain: **76.6 V/(m/s)**
- Overall sensitivity: **76.6**

If your data is in mV and you use this RESP:

$$\text{velocity} = \frac{\text{data [mV]}}{76.6 \text{ [V/(m/s)]}} = \frac{\text{data} \times 10^{-3} \text{ [V]}}{76.6 \text{ [V/(m/s)]}}$$

You'd get values that are **1000× too large** because the RESP assumes the input is in Volts, but it's actually in millivolts.

### The Fix: fromMilliVolts RESP

The `RESP.XX.NS680..SPZ.DTSOLO.5.1850.43000.76_6_fromMilliVolts` file corrects for this:
- Stage 1 gain: **76,600 mV/(m/s)** = 76.6 V/(m/s) × 1000
- Overall sensitivity: **76,600**

$$\text{velocity} = \frac{\text{data [mV]}}{76600 \text{ [mV/(m/s)]}} = \text{correct m/s}$$

### Quick Check

You can verify which RESP to use by checking the FDSN XML sensitivity:
- FDSN XML says: sensitivity = **7.68e4** mV/(m/s) ≈ 76,800 mV/(m/s)
- fromMilliVolts RESP: sensitivity = **76,600** mV/(m/s)
- Standard RESP: sensitivity = **76.6** V/(m/s)

The small difference (76,600 vs 76,800) likely reflects slightly different calibration values — both are within the typical tolerance of the geophone.

In [ ]:
# Demonstrate the ×1000 error with a synthetic example

# Simulate a 5 Hz signal in mV (as SmartSolo would output)
np.random.seed(42)
npts = 25000  # 100 seconds at 250 sps
dt = 1.0 / 250.0
t = np.arange(npts) * dt

# Fake mV signal: a 5 Hz sine wave with amplitude ~100 mV
fake_mv = 100 * np.sin(2 * np.pi * 5 * t) + np.random.randn(npts) * 5

# "Correct" velocity using fromMilliVolts RESP (sensitivity = 76,600)
velocity_correct = fake_mv / 76600  # m/s

# "Wrong" velocity using standard RESP (sensitivity = 76.6)
velocity_wrong = fake_mv / 76.6  # m/s — 1000× too large!

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

t_slice = slice(0, 2500)  # first 10 seconds
ax1.plot(t[t_slice], velocity_correct[t_slice] * 1e6, 'g-', linewidth=1, label='Correct (fromMilliVolts RESP)')
ax1.set_ylabel('Velocity [µm/s]')
ax1.set_title('Response removal with CORRECT RESP (fromMilliVolts)')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(t[t_slice], velocity_wrong[t_slice] * 1e6, 'r-', linewidth=1, label='WRONG (standard V RESP)')
ax2.set_ylabel('Velocity [µm/s]')
ax2.set_xlabel('Time [s]')
ax2.set_title('Response removal with WRONG RESP — amplitudes 1000× too large!')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Correct peak velocity:  {np.max(np.abs(velocity_correct))*1e6:.2f} µm/s")
print(f"Wrong peak velocity:    {np.max(np.abs(velocity_wrong))*1e6:.2f} µm/s")
print(f"Ratio (wrong/correct):  {np.max(np.abs(velocity_wrong))/np.max(np.abs(velocity_correct)):.0f}×")

<a id="part2-5-pipeline"></a>
## Part 2.5 — Full Preprocessing Pipeline

Here's the standard preprocessing workflow applied to real SmartSolo data.

In [ ]:
# Load a fresh copy of the data
st_raw = read(files_5hz_z[1])  # One day of 5 Hz station vertical

print(f"Loaded: {st_raw[0].stats.network}.{st_raw[0].stats.station} "
      f"{st_raw[0].stats.channel}")
print(f"Start: {st_raw[0].stats.starttime}")
print(f"End:   {st_raw[0].stats.endtime}")
print(f"Npts:  {st_raw[0].stats.npts}")
print(f"Sampling rate: {st_raw[0].stats.sampling_rate} Hz")
print(f"Duration: {st_raw[0].stats.npts / st_raw[0].stats.sampling_rate:.1f} seconds "
      f"({st_raw[0].stats.npts / st_raw[0].stats.sampling_rate / 3600:.1f} hours)")

In [ ]:
# Step-by-step preprocessing with visualization at each stage

# Work with a 5-minute window for clarity
t_start = st_raw[0].stats.starttime
window_start = t_start + 3600  # Start 1 hour into the record
window_end = window_start + 300  # 5 minutes

fig, axes = plt.subplots(5, 1, figsize=(14, 16), sharex=True)

# Step 0: Raw
tr = st_raw[0].copy().slice(window_start, window_end)
axes[0].plot(tr.times(), tr.data, 'k-', linewidth=0.3)
axes[0].set_title('Step 0: Raw data (mV from SmartSolo)')
axes[0].set_ylabel('mV')
axes[0].grid(True, alpha=0.3)
axes[0].text(0.02, 0.95, f'Mean: {tr.data.mean():.1f}', transform=axes[0].transAxes, 
             va='top', fontsize=10, bbox=dict(boxstyle='round', facecolor='wheat'))

# Step 1: Demean
tr1 = st_raw[0].copy().slice(window_start, window_end)
tr1.detrend('demean')
axes[1].plot(tr1.times(), tr1.data, 'b-', linewidth=0.3)
axes[1].set_title('Step 1: After demean')
axes[1].set_ylabel('mV')
axes[1].grid(True, alpha=0.3)
axes[1].text(0.02, 0.95, f'Mean: {tr1.data.mean():.2e}', transform=axes[1].transAxes, 
             va='top', fontsize=10, bbox=dict(boxstyle='round', facecolor='wheat'))

# Step 2: Linear detrend
tr2 = st_raw[0].copy().slice(window_start, window_end)
tr2.detrend('linear')
axes[2].plot(tr2.times(), tr2.data, 'g-', linewidth=0.3)
axes[2].set_title('Step 2: After linear detrend (removes trend + mean)')
axes[2].set_ylabel('mV')
axes[2].grid(True, alpha=0.3)

# Step 3: Taper
tr3 = st_raw[0].copy().slice(window_start, window_end)
tr3.detrend('linear')
tr3.taper(max_percentage=0.05, type='cosine')
axes[3].plot(tr3.times(), tr3.data, 'orange', linewidth=0.3)
axes[3].set_title('Step 3: After detrend + 5% cosine taper')
axes[3].set_ylabel('mV')
axes[3].grid(True, alpha=0.3)

# Step 4: Bandpass filter
tr4 = st_raw[0].copy().slice(window_start, window_end)
tr4.detrend('linear')
tr4.taper(max_percentage=0.05, type='cosine')
tr4.filter('bandpass', freqmin=1.0, freqmax=50.0, corners=4, zerophase=True)
axes[4].plot(tr4.times(), tr4.data, 'r-', linewidth=0.3)
axes[4].set_title('Step 4: After detrend + taper + bandpass 1–50 Hz')
axes[4].set_ylabel('mV')
axes[4].set_xlabel('Time [s]')
axes[4].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Compare spectra at each preprocessing step

fig, ax = plt.subplots(figsize=(14, 6))

for tr_step, label, color in [
    (tr, 'Raw', 'black'),
    (tr1, 'Demeaned', 'blue'),
    (tr2, 'Detrended', 'green'),
    (tr3, 'Detrended + Tapered', 'orange'),
    (tr4, 'Detrended + Tapered + Filtered', 'red')]:
    
    nfft = len(tr_step.data)
    freqs_fft = np.fft.rfftfreq(nfft, d=tr_step.stats.delta)
    spectrum = np.abs(np.fft.rfft(tr_step.data))
    
    ax.loglog(freqs_fft[1:], spectrum[1:], color=color, linewidth=1, alpha=0.7, label=label)

ax.set_xlabel('Frequency [Hz]')
ax.set_ylabel('Spectral Amplitude')
ax.set_title('Effect of Each Preprocessing Step on the Spectrum')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
ax.set_xlim(0.01, 125)

plt.tight_layout()
plt.show()

print("Key observations:")
print("- Demean: removes the DC spike (leftmost peak)")
print("- Detrend: further reduces very low frequency energy")
print("- Taper: smooths the spectrum (reduces spectral leakage artifacts)")
print("- Filter: sharp cutoff outside 1–50 Hz band")

<a id="part2-6-removal"></a>
## Part 2.6 — Response Removal in Practice

Now let's remove the instrument response from real SmartSolo data using ObsPy.

In [ ]:
# Method 1: Using RESP file with remove_response()
# Note: ObsPy's remove_response needs an Inventory object

# Load the fromMilliVolts RESP as inventory
inv_resp = read_inventory(RESP_MILLIVOLTS, format='RESP')

# The RESP file uses station XX.NS680, channel SPZ
# Our data uses SS.{station}.SW.DPZ — we need to match them
# Option A: Modify the inventory metadata to match our data
# Option B: Use the FDSN XML from the SmartSolo export
# Option C: Use evalresp with the raw RESP file
# Option D: Build a PAZ dictionary manually

# Let's use the cleanest approach: build PAZ and use simulate_seismometer

# SmartSolo 5 Hz PAZ (in millivolts)
paz_5hz_mv = {
    'poles': [(-22.211059+22.217768j), (-22.211059-22.217768j)],
    'zeros': [0j, 0j],
    'gain': 1.0,           # A0 normalization factor
    'sensitivity': 76600   # Overall sensitivity in mV/(m/s)
}

# SmartSolo 2 Hz PAZ (in millivolts — scale sensitivity ×1000)
paz_2hz_mv = {
    'poles': [(-8.8844236+8.8871072j), (-8.8844236-8.8871072j)],
    'zeros': [0j, 0j],
    'gain': 1.0,
    'sensitivity': 260000  # 260 V/(m/s) × 1000 = 260,000 mV/(m/s)
}

print("PAZ dictionaries ready for response removal.")
print(f"\n5 Hz sensitivity: {paz_5hz_mv['sensitivity']} mV/(m/s)")
print(f"2 Hz sensitivity: {paz_2hz_mv['sensitivity']} mV/(m/s)")

In [ ]:
# Response removal using ObsPy's simulate_seismometer
from obspy.signal.invsim import simulate_seismometer

# Process 5 Hz station data
st_5hz_proc = read(files_5hz_z[1])
tr_5hz_vel = st_5hz_proc[0].copy()

# Trim to a manageable window
t0 = tr_5hz_vel.stats.starttime + 3600  # 1 hour in
tr_5hz_vel = tr_5hz_vel.slice(t0, t0 + 300)  # 5 minutes

# Step 1: Detrend
tr_5hz_vel.detrend('linear')

# Step 2: Taper
tr_5hz_vel.taper(max_percentage=0.05, type='cosine')

# Step 3: Remove response using simulate_seismometer
pre_filt = [0.5, 1.0, 100, 120]  # f1, f2, f3, f4

tr_5hz_vel.simulate(
    paz_remove=paz_5hz_mv,
    paz_simulate=None,     # None = output in ground motion units
    pre_filt=pre_filt,
    taper=True,
    taper_fraction=0.05
)

print(f"After response removal:")
print(f"  Data range: {tr_5hz_vel.data.min():.2e} to {tr_5hz_vel.data.max():.2e} m/s")
print(f"  Peak velocity: {np.max(np.abs(tr_5hz_vel.data))*1e6:.2f} µm/s")

In [ ]:
# Compare raw vs response-corrected

# Reload raw for comparison
st_raw_cmp = read(files_5hz_z[1])
tr_raw_cmp = st_raw_cmp[0].copy().slice(t0, t0 + 300)
tr_raw_cmp.detrend('linear')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Zoom into 30 seconds of interesting data
ax1.plot(tr_raw_cmp.times(), tr_raw_cmp.data, 'k-', linewidth=0.5)
ax1.set_title(f'Raw Data (detrended) — {STATION_5HZ} DPZ (5 Hz geophone)')
ax1.set_ylabel('mV')
ax1.grid(True, alpha=0.3)

ax2.plot(tr_5hz_vel.times(), tr_5hz_vel.data * 1e6, 'r-', linewidth=0.5)
ax2.set_title('After Response Removal — True Ground Velocity')
ax2.set_ylabel('Velocity [µm/s]')
ax2.set_xlabel('Time [s]')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Method 2: Using FDSN StationXML with remove_response()
# This is the "modern" approach when you have StationXML files

# Load the FDSN XML (exported by SmartSolo)
fdsn_files = sorted(glob.glob(f'{FDSN_DIR_2HZ}/*.xml'))

if fdsn_files:
    inv_fdsn = read_inventory(fdsn_files[0])
    print("Loaded FDSN StationXML:")
    print(inv_fdsn)
    
    # Check what station this is for
    net = inv_fdsn[0]
    sta = net[0]
    cha = sta[0]
    print(f"\nNetwork: {net.code}")
    print(f"Station: {sta.code}")
    print(f"Channel: {cha.code}")
    print(f"Location: {cha.location_code}")
    print(f"\nSensitivity: {cha.response.instrument_sensitivity.value}")
    print(f"Input units: {cha.response.instrument_sensitivity.input_units}")
    print(f"Output units: {cha.response.instrument_sensitivity.output_units}")
    print(f"\nCoordinates: {sta.latitude}°N, {sta.longitude}°E, {sta.elevation}m")
else:
    print("No FDSN XML files found.")

In [ ]:
# Method 2 continued: remove_response using the FDSN inventory

if fdsn_files:
    # Load data for the station matching the FDSN XML
    sta_code = inv_fdsn[0][0].code
    matching_files = sorted(glob.glob(f'{MSEED_2HZ_DIR}/SS.{sta_code}.SW.DPZ.*.miniseed'))
    
    if matching_files:
        st_fdsn = read(matching_files[1])  # Second file (full day)
        tr_fdsn = st_fdsn[0].copy()
        
        # Trim to 5 minutes
        t0_fdsn = tr_fdsn.stats.starttime + 3600
        tr_fdsn = tr_fdsn.slice(t0_fdsn, t0_fdsn + 300)
        
        # Preprocessing
        tr_fdsn.detrend('linear')
        tr_fdsn.taper(max_percentage=0.05, type='cosine')
        
        # Remove response using the FDSN inventory
        pre_filt_2hz = [0.2, 0.5, 100, 120]
        tr_fdsn.remove_response(
            inventory=inv_fdsn,
            output='VEL',
            pre_filt=pre_filt_2hz
        )
        
        print(f"Response removed using FDSN XML for station {sta_code}")
        print(f"Peak velocity: {np.max(np.abs(tr_fdsn.data))*1e6:.2f} µm/s")
        
        # Plot
        fig, ax = plt.subplots(figsize=(14, 4))
        ax.plot(tr_fdsn.times(), tr_fdsn.data * 1e6, 'b-', linewidth=0.5)
        ax.set_title(f'2 Hz Station {sta_code} — Response Removed (FDSN XML method)')
        ax.set_ylabel('Velocity [µm/s]')
        ax.set_xlabel('Time [s]')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

In [ ]:
# Method 3: Using SAC Pole-Zero files
# ObsPy can read SAC PZ files directly

from obspy.io.sac.sacpz import attach_paz

# Load 5 Hz data
st_pz = read(files_5hz_z[1])
tr_pz = st_pz[0].copy()
t0_pz = tr_pz.stats.starttime + 3600
tr_pz = tr_pz.slice(t0_pz, t0_pz + 300)

# Attach PAZ from the .pz file
# Note: the PZ file has CONSTANT = 76.7 (in V/(m/s))
# For mV data, we need to multiply by 1000
attach_paz(tr_pz, f'{RESP_DIR}/smartsolo.5Hz.pz')

# Check what was attached
print("PAZ attached from .pz file:")
print(f"  Poles: {tr_pz.stats.paz.poles}")
print(f"  Zeros: {tr_pz.stats.paz.zeros}")
print(f"  Gain: {tr_pz.stats.paz.gain}")
print(f"  Sensitivity: {tr_pz.stats.paz.sensitivity}")

print("\n⚠️  WARNING: This PZ file uses V/(m/s) units.")
print("   For SmartSolo mV data, multiply sensitivity by 1000.")
print(f"   Corrected sensitivity should be: {tr_pz.stats.paz.sensitivity * 1000}")

In [ ]:
# Batch processing: remove response for all 3 components

def remove_response_smartsolo(mseed_file, paz_dict, pre_filt, output='VEL',
                                trim_start=None, trim_end=None):
    """
    Load a SmartSolo MiniSEED file, preprocess, and remove instrument response.
    
    Parameters
    ----------
    mseed_file : str
        Path to MiniSEED file
    paz_dict : dict
        Poles-and-zeros dictionary with 'poles', 'zeros', 'gain', 'sensitivity'
    pre_filt : list
        [f1, f2, f3, f4] corner frequencies for pre-filter
    output : str
        Output type: 'VEL', 'DISP', or 'ACC'
    trim_start, trim_end : UTCDateTime, optional
        Trim window
    
    Returns
    -------
    obspy.Trace
        Processed trace in physical units
    """
    tr = read(mseed_file)[0]
    
    if trim_start and trim_end:
        tr = tr.slice(trim_start, trim_end)
    
    # Preprocessing
    tr.detrend('linear')
    tr.taper(max_percentage=0.05, type='cosine')
    
    # Response removal
    tr.simulate(
        paz_remove=paz_dict,
        paz_simulate=None,
        pre_filt=pre_filt
    )
    
    return tr

# Process all 3 components for the 5 Hz station
pre_filt_5hz = [0.5, 1.0, 100, 120]
t0 = UTCDateTime(2026, 8, 30, 1, 0, 0)  # 1 AM on day 2
t1 = t0 + 300  # 5 minutes

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

for idx, (comp, comp_name) in enumerate([('DPZ', 'Vertical'), ('DPN', 'North'), ('DPE', 'East')]):
    files_comp = sorted(glob.glob(f'{MSEED_5HZ_DIR}/SS.{STATION_5HZ}.SW.{comp}.*.miniseed'))
    if len(files_comp) > 1:
        tr_comp = remove_response_smartsolo(
            files_comp[1], paz_5hz_mv, pre_filt_5hz,
            trim_start=t0, trim_end=t1
        )
        axes[idx].plot(tr_comp.times(), tr_comp.data * 1e6, linewidth=0.5)
        axes[idx].set_ylabel(f'{comp_name}\n[µm/s]')
        axes[idx].set_title(f'{comp} — {comp_name} Component')
        axes[idx].grid(True, alpha=0.3)

axes[-1].set_xlabel('Time [s]')
fig.suptitle(f'Station {STATION_5HZ} — All Components After Response Removal', y=1.02)
plt.tight_layout()
plt.show()

<a id="part2-7-polarity"></a>
## Part 2.7 — Polarity Issues with SmartSolo Nodes

### The Problem

SmartSolo nodes have a **known polarity issue** documented by AusPass (Australian Passive Seismic Server):

> SmartSolo nodes can have **reversed polarity** on some or all components. This means the recorded waveform is "upside down" — a compression (downward first motion on a vertical seismometer) appears as a dilatation (upward first motion), and vice versa.

### Why Does It Matter?

- **First motion studies**: P-wave polarity is critical for focal mechanism determination
- **Receiver functions**: Reversed polarity gives wrong Moho depth estimates
- **Cross-correlation**: Reversed polarity produces anti-correlation instead of correlation
- **Waveform stacking**: Reversed traces destructively cancel the signal

### How to Detect It

1. **Teleseismic test**: Record a known teleseismic event (e.g., from USGS) and compare the P-wave first motion with the expected polarity from the focal mechanism.

2. **Comparison with reference station**: If you have a broadband station nearby, compare waveforms — they should have the same polarity.

3. **Check the `IGU-BD3C-5_PZ_Pole.pz` file**: The CONSTANT is **negative** (-3.554090e-03), which encodes a polarity reversal in the response itself.

### How to Fix It

**Option 1**: Multiply the data by -1:
```python
tr.data *= -1
```

**Option 2**: Use a response file with negative gain (like `IGU-BD3C-5_PZ_Pole.pz` which has negative CONSTANT).

**Important**: Document which approach you use and apply it consistently across your dataset!

In [ ]:
# Demonstrate polarity reversal effect

# Load a trace
st_pol = read(files_5hz_z[1])
tr_pol = st_pol[0].copy()
t0_pol = tr_pol.stats.starttime + 7200  # 2 hours in
tr_pol = tr_pol.slice(t0_pol, t0_pol + 60)  # 60 seconds
tr_pol.detrend('linear')

# Create reversed version
tr_reversed = tr_pol.copy()
tr_reversed.data *= -1

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

ax1.plot(tr_pol.times(), tr_pol.data, 'b-', linewidth=0.5, label='Normal polarity')
ax1.set_title('Normal Polarity')
ax1.set_ylabel('Amplitude')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(tr_reversed.times(), tr_reversed.data, 'r-', linewidth=0.5, label='Reversed polarity (×-1)')
ax2.set_title('Reversed Polarity — Everything is Flipped!')
ax2.set_ylabel('Amplitude')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Show cross-correlation to detect polarity
cc = np.correlate(tr_pol.data, tr_reversed.data, mode='full')
lags = np.arange(-len(tr_pol.data)+1, len(tr_pol.data))
ax3.plot(lags / tr_pol.stats.sampling_rate, cc / np.max(np.abs(cc)), 'k-', linewidth=0.5)
ax3.axhline(0, color='gray', linestyle='--')
ax3.set_title('Cross-correlation: Normal vs Reversed → Peak is NEGATIVE (anti-correlation)')
ax3.set_ylabel('Normalized CC')
ax3.set_xlabel('Lag [s]')
ax3.grid(True, alpha=0.3)

peak_cc = cc[len(tr_pol.data)-1] / np.max(np.abs(cc))
ax3.annotate(f'Zero-lag CC = {peak_cc:.2f}', xy=(0, peak_cc), fontsize=12,
             xytext=(5, 0.5), arrowprops=dict(arrowstyle='->', color='red'),
             color='red', fontsize=12)

plt.tight_layout()
plt.show()

print(f"Zero-lag cross-correlation coefficient: {peak_cc:.2f}")
print("A value of -1.0 confirms perfect polarity reversal.")

In [ ]:
# Check the IGU-BD3C-5 PZ file — it has a negative CONSTANT
# This encodes the polarity reversal directly in the response

pz_file = f'{RESP_DIR}/IGU-BD3C-5_PZ_Pole.pz'

print("=== IGU-BD3C-5_PZ_Pole.pz ===")
print("This PZ file includes the full analog electronics response")
print("(not just the geophone, but also anti-alias filters)\n")

# Parse it
zeros_igu = [
    complex(1.4164e4, 0),
    complex(-7.162e3, 0),
    complex(0, 0),
    complex(0, 0)
]
poles_igu = [
    complex(-1.7204e3, 0),
    complex(-1.2, 9.0e-1),
    complex(-1.2, -9.0e-1)
]
constant_igu = -3.554090e-03

print(f"Zeros: {zeros_igu}")
print(f"Poles: {poles_igu}")
print(f"CONSTANT: {constant_igu}")
print(f"\n→ The NEGATIVE constant means polarity is reversed in this response.")
print(f"  Using this PZ file for response removal will automatically")
print(f"  correct the polarity reversal.")

# Plot this more complex response
freqs_igu = np.logspace(-1, 2.3, 1000)
resp_igu = compute_response(freqs_igu, zeros_igu, poles_igu, np.abs(constant_igu))

fig, ax = plt.subplots(figsize=(14, 5))
ax.loglog(freqs_igu, np.abs(resp_igu), 'purple', linewidth=2, label='IGU-BD3C-5 full response')
ax.loglog(freqs_igu, np.abs(compute_response(freqs_igu, zeros_5hz, poles_5hz, gain_5hz)), 
          'b--', linewidth=2, alpha=0.5, label='Simple 5 Hz geophone only')
ax.set_xlabel('Frequency [Hz]')
ax.set_ylabel('Amplitude')
ax.set_title('IGU-BD3C-5 Full Response vs Simple Geophone Response')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

print("\nThe full response includes additional zeros and poles from the electronics,")
print("which create a more complex transfer function shape at high frequencies.")

<a id="part2-8-exercises"></a>
## Part 2.8 — Exercises

Test your understanding with these exercises. Answers are provided in hidden cells below each exercise.

### Exercise 1: Compute Natural Frequency from Poles

Given the poles `(-8.8844236 + 8.8871072j)` and `(-8.8844236 - 8.8871072j)`, compute:

1. The natural frequency $f_0$ in Hz
2. The damping ratio $h$
3. The angular natural frequency $\omega_0$ in rad/s

In [ ]:
# YOUR ANSWER HERE
# ================
# Given:
pole = -8.8844236 + 8.8871072j

# Compute omega_0:
omega_0 = ...  # YOUR CODE

# Compute f_0:
f_0 = ...  # YOUR CODE

# Compute damping ratio:
h = ...  # YOUR CODE

print(f"omega_0 = {omega_0:.4f} rad/s")
print(f"f_0 = {f_0:.4f} Hz")
print(f"h = {h:.4f}")

In [ ]:
# SOLUTION — Exercise 1
# ======================

pole = -8.8844236 + 8.8871072j

sigma = pole.real      # -8.8844236
omega_d = pole.imag    # 8.8871072 (damped natural frequency)

# Natural angular frequency
omega_0 = np.sqrt(sigma**2 + omega_d**2)
print(f"omega_0 = sqrt({sigma:.4f}² + {omega_d:.4f}²) = {omega_0:.4f} rad/s")

# Natural frequency in Hz
f_0 = omega_0 / (2 * np.pi)
print(f"f_0 = {omega_0:.4f} / (2π) = {f_0:.4f} Hz")

# Damping ratio
h = -sigma / omega_0
print(f"h = -{sigma:.4f} / {omega_0:.4f} = {h:.4f}")
print(f"\nDamping ratio ≈ {h:.2f} → {'underdamped' if h < 1 else 'overdamped'}")
print(f"(0.707 = critical damping for a Butterworth-like response)")

### Exercise 2: Which RESP File Should You Use?

You have SmartSolo IGU-16HR 3C 5Hz data. The FDSN XML says the output is in millivolts.

You have two RESP files:
- `DT-SOLO.5Hz.RESP` with sensitivity = 76.6
- `RESP...fromMilliVolts` with sensitivity = 76,600

1. Which one should you use and why?
2. If you accidentally use the wrong one, by what factor will your velocity estimates be off?
3. How would you verify you used the correct one?

In [ ]:
# YOUR ANSWER HERE — write your reasoning as comments:
#
# 1. Which RESP to use?
#    ...
#
# 2. Error factor if wrong?
#    ...
#
# 3. How to verify?
#    ...

In [ ]:
# SOLUTION — Exercise 2
# ======================

print("""1. Which RESP to use?
   → Use the 'fromMilliVolts' RESP (sensitivity = 76,600 mV/(m/s))
   Because SmartSolo outputs data in millivolts (mV), not Volts.
   The FDSN XML confirms: <OutputUnits><Name>mV</Name></OutputUnits>

2. Error factor if wrong?
   → If you use the standard RESP (76.6 V/(m/s)) with mV data:
   velocity = data_mV / 76.6 = (data_V × 1000) / 76.6
   This is 1000× too large.
   
   Factor = 76600 / 76.6 = 1000×
   Your velocity estimates would be 1000× TOO LARGE.

3. How to verify?
   a) Compare with known earthquake amplitudes:
      - Ambient noise: ~0.01-1 µm/s
      - Local M2 earthquake at 50 km: ~10-100 µm/s
      - If you're getting mm/s for ambient noise, something is wrong.
   
   b) Compare with a nearby reference station that has known calibration.
   
   c) Check the FDSN XML sensitivity value (7.68e4) matches your RESP.""")

# Quantitative check
ratio = 76600 / 76.6
print(f"\nRatio = {ratio:.0f}× — exactly 1000 as expected.")

### Exercise 3: Pre-processing Order Matters

Below are four different preprocessing orders. Which one is correct and why are the others wrong?

- A) `taper → detrend → filter → remove_response`
- B) `detrend → taper → remove_response → filter`
- C) `remove_response → detrend → taper → filter`
- D) `filter → detrend → taper → remove_response`

In [ ]:
# YOUR ANSWER HERE:
# ...

In [ ]:
# SOLUTION — Exercise 3
# ======================

print("""CORRECT: B) detrend → taper → remove_response → filter

Why the others are wrong:

A) taper → detrend → ...
   WRONG: If you taper BEFORE detrend, the taper introduces new trends at
   the edges (because the non-zero mean/trend is being windowed). Always
   detrend first, then taper.

C) remove_response → detrend → ...
   WRONG: Response removal before detrend will amplify the trend/offset
   at low frequencies (where the response → 0), producing massive artifacts.
   You MUST detrend before response removal.

D) filter → detrend → ...
   PROBLEMATIC: Filtering before detrend can work but is suboptimal —
   the filter itself can introduce edge effects on data with trends.
   Also, filtering before response removal changes the frequency content,
   which can interact badly with the pre_filt in remove_response.

BEST PRACTICE:
   1. detrend (remove trend and mean)
   2. taper (smooth edges to zero)
   3. remove_response (with pre_filt to handle bandwidth limiting)
   4. filter (optional, for further frequency selection)

Note: If you use pre_filt in remove_response, you may not need step 4.
""")

### Exercise 4: Hands-On — Process a Different Station

Using the functions we've built above:

1. Pick a different 2 Hz station from the dataset
2. Load one day of the vertical (DPZ) component
3. Apply the full preprocessing pipeline
4. Remove the instrument response using the 2 Hz PAZ dictionary
5. Plot the result and check if the amplitudes are reasonable

**Hint**: Use `paz_2hz_mv` for the response removal, and `pre_filt = [0.2, 0.5, 100, 120]` for the 2 Hz geophone.

In [ ]:
# YOUR CODE HERE
# ==============

# 1. Pick a station
my_station = '...'  # e.g., '00011', '00017', '00023', etc.

# 2. Find and load the file
# ...

# 3-4. Preprocess and remove response
# ...

# 5. Plot
# ...

In [ ]:
# SOLUTION — Exercise 4
# ======================

# 1. Pick a station
my_station = '00011'

# 2. Find and load the file
my_files = sorted(glob.glob(f'{MSEED_2HZ_DIR}/SS.{my_station}.SW.DPZ.*.miniseed'))
print(f"Found {len(my_files)} files for station {my_station}")

if len(my_files) > 1:
    # 3-4. Process using our function
    t_start_ex = UTCDateTime(2026, 8, 30, 6, 0, 0)  # 6 AM
    t_end_ex = t_start_ex + 600  # 10 minutes
    
    tr_ex = remove_response_smartsolo(
        my_files[1],
        paz_2hz_mv,  # 2 Hz PAZ in millivolts
        pre_filt=[0.2, 0.5, 100, 120],
        trim_start=t_start_ex,
        trim_end=t_end_ex
    )
    
    # 5. Plot
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(tr_ex.times(), tr_ex.data * 1e6, 'b-', linewidth=0.5)
    ax.set_title(f'Station {my_station} (2 Hz) — DPZ — Response Removed')
    ax.set_ylabel('Velocity [µm/s]')
    ax.set_xlabel('Time [s]')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Sanity check
    peak_vel = np.max(np.abs(tr_ex.data)) * 1e6
    rms_vel = np.std(tr_ex.data) * 1e6
    print(f"\nSanity check:")
    print(f"  Peak velocity: {peak_vel:.2f} µm/s")
    print(f"  RMS velocity:  {rms_vel:.2f} µm/s")
    print(f"  Expected for ambient noise: 0.01-1 µm/s")
    print(f"  → {'Looks reasonable!' if 0.001 < rms_vel < 100 else 'Check your parameters!'}")
else:
    print(f"No files found for station {my_station}. Try a different station.")

### Exercise 5: Read a RESP File Manually

Without using ObsPy, parse the RESP file below and extract:
1. The poles and zeros
2. The normalization factor
3. The stage gains
4. The overall sensitivity
5. Compute the natural frequency from the poles

In [ ]:
# Here's the RESP file content as a string — parse it!

resp_text = """
B053F03     Transfer function type:                A [Laplace Transform (Rad/sec)]
B053F04     Stage sequence number:                 1
B053F05     Response in units lookup:              M/S - Velocity in Meters Per Second
B053F06     Response out units lookup:             V - Volts
B053F07     A0 normalization factor:               1.000000E+00
B053F08     Normalization frequency:               5.000000E+01
B053F09     Number of zeroes:                      2
B053F14     Number of poles:                       2
B053F10-13    0  0.000000e+00  0.000000e+00
B053F10-13    1  0.000000e+00  0.000000e+00
B053F15-18    0 -2.221110e+01 -2.221780e+01
B053F15-18    1 -2.221110e+01  2.221780e+01
B058F04     Gain:                                  7.660000e+04
B058F05     Frequency of gain:                     5.000000e+01 HZ
B058F03     Stage sequence number:                 2
B058F04     Gain:                                  1.000000e+00
B058F03     Stage sequence number:                 0
B058F04     Sensitivity:                           7.660000e+04
"""

# YOUR CODE: Parse this and extract the values
# ...

In [ ]:
# SOLUTION — Exercise 5
# ======================

import re

# Parse zeros
zero_pattern = r'B053F10-13\s+\d+\s+([\d.eE+-]+)\s+([\d.eE+-]+)'
zeros_parsed = [complex(float(m[0]), float(m[1])) 
                for m in re.findall(zero_pattern, resp_text)]

# Parse poles  
pole_pattern = r'B053F15-18\s+\d+\s+([\d.eE+-]+)\s+([\d.eE+-]+)'
poles_parsed = [complex(float(m[0]), float(m[1])) 
                for m in re.findall(pole_pattern, resp_text)]

# Parse A0
a0_match = re.search(r'A0 normalization factor:\s+([\d.eE+-]+)', resp_text)
a0_parsed = float(a0_match.group(1))

# Parse gains
gain_pattern = r'B058F04\s+(?:Gain|Sensitivity):\s+([\d.eE+-]+)'
gains = [float(g) for g in re.findall(gain_pattern, resp_text)]

print("Parsed RESP file:")
print(f"  Zeros: {zeros_parsed}")
print(f"  Poles: {poles_parsed}")
print(f"  A0 (normalization): {a0_parsed}")
print(f"  Stage 1 gain: {gains[0]}")
print(f"  Stage 2 gain: {gains[1]}")
print(f"  Overall sensitivity: {gains[2]}")
print(f"  Sensitivity check: {gains[0]} × {gains[1]} = {gains[0]*gains[1]} (should equal {gains[2]})")

# Natural frequency from poles
p = poles_parsed[0]
omega_0_ex = np.sqrt(p.real**2 + p.imag**2)
f_0_ex = omega_0_ex / (2 * np.pi)
h_ex = -p.real / omega_0_ex

print(f"\nDerived quantities:")
print(f"  Natural frequency: {f_0_ex:.2f} Hz")
print(f"  Damping ratio: {h_ex:.4f}")
print(f"  This is the fromMilliVolts RESP (gain = 76,600 mV/(m/s))")

### Exercise 6: Compare Response Removal Methods

Apply response removal to the same data using three methods:
1. PAZ dictionary with `simulate()`
2. FDSN StationXML with `remove_response()`
3. RESP file with `remove_response()`

Compare the results — they should be nearly identical.

In [ ]:
# SOLUTION — Exercise 6 (also serves as a comprehensive comparison)
# =================================================================

# We'll use a 2 Hz station that has FDSN XML available

if fdsn_files:
    sta_code_ex6 = inv_fdsn[0][0].code
    files_ex6 = sorted(glob.glob(f'{MSEED_2HZ_DIR}/SS.{sta_code_ex6}.SW.DPZ.*.miniseed'))
    
    if len(files_ex6) > 1:
        t0_ex6 = UTCDateTime(2026, 8, 30, 12, 0, 0)  # noon
        t1_ex6 = t0_ex6 + 300  # 5 minutes
        pre_filt_ex6 = [0.2, 0.5, 100, 120]
        
        # Method 1: PAZ + simulate()
        tr_m1 = read(files_ex6[1])[0].copy().slice(t0_ex6, t1_ex6)
        tr_m1.detrend('linear')
        tr_m1.taper(max_percentage=0.05, type='cosine')
        tr_m1.simulate(paz_remove=paz_2hz_mv, paz_simulate=None, pre_filt=pre_filt_ex6)
        
        # Method 2: FDSN XML + remove_response()
        tr_m2 = read(files_ex6[1])[0].copy().slice(t0_ex6, t1_ex6)
        tr_m2.detrend('linear')
        tr_m2.taper(max_percentage=0.05, type='cosine')
        tr_m2.remove_response(inventory=inv_fdsn, output='VEL', pre_filt=pre_filt_ex6)
        
        # Compare
        fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
        
        axes[0].plot(tr_m1.times(), tr_m1.data * 1e6, 'b-', linewidth=0.5)
        axes[0].set_title('Method 1: PAZ dictionary + simulate()')
        axes[0].set_ylabel('Velocity [µm/s]')
        axes[0].grid(True, alpha=0.3)
        
        axes[1].plot(tr_m2.times(), tr_m2.data * 1e6, 'r-', linewidth=0.5)
        axes[1].set_title('Method 2: FDSN StationXML + remove_response()')
        axes[1].set_ylabel('Velocity [µm/s]')
        axes[1].grid(True, alpha=0.3)
        
        # Difference
        # Resample if needed to match lengths
        min_len = min(len(tr_m1.data), len(tr_m2.data))
        diff = (tr_m1.data[:min_len] - tr_m2.data[:min_len]) * 1e6
        axes[2].plot(tr_m1.times()[:min_len], diff, 'k-', linewidth=0.5)
        axes[2].set_title(f'Difference (Method 1 - Method 2) — Max: {np.max(np.abs(diff)):.4f} µm/s')
        axes[2].set_ylabel('Diff [µm/s]')
        axes[2].set_xlabel('Time [s]')
        axes[2].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative comparison
        cc_coeff = np.corrcoef(tr_m1.data[:min_len], tr_m2.data[:min_len])[0, 1]
        amp_ratio = np.std(tr_m1.data[:min_len]) / np.std(tr_m2.data[:min_len])
        print(f"\nCorrelation between methods: {cc_coeff:.6f}")
        print(f"Amplitude ratio (M1/M2): {amp_ratio:.4f}")
        print(f"\nSmall differences are expected due to:")
        print(f"  - Slightly different sensitivity values (76,600 vs 76,800)")
        print(f"  - Different implementation details in simulate() vs remove_response()")
else:
    print("No FDSN XML files available for this comparison.")
    print("Run this on the server where the FDSN Information directory is accessible.")

---
## Summary & Best Practices

### Preprocessing Workflow for SmartSolo Data

```python
import obspy

# 1. Load data
st = obspy.read('SS.06384.SW.DPZ.*.miniseed')

# 2. Detrend (removes mean + linear trend)
st.detrend('linear')

# 3. Taper (smooth edges to zero)
st.taper(max_percentage=0.05, type='cosine')

# 4. Remove response (with frequency-domain pre-filter)
paz_5hz_mv = {
    'poles': [(-22.211059+22.217768j), (-22.211059-22.217768j)],
    'zeros': [0j, 0j],
    'gain': 1.0,
    'sensitivity': 76600  # mV/(m/s) — USE fromMilliVolts!
}
pre_filt = [0.5, 1.0, 100, 120]

for tr in st:
    tr.simulate(paz_remove=paz_5hz_mv, pre_filt=pre_filt)

# 5. (Optional) Additional filtering
st.filter('bandpass', freqmin=1, freqmax=50, zerophase=True)
```

### Key Takeaways

1. **MiniSEED stores raw values** (counts or mV for SmartSolo) — not physical units
2. **SmartSolo outputs in millivolts** — use the `fromMilliVolts` RESP (sensitivity ≈ 76,600) 
3. **Always detrend before response removal** — or face catastrophic low-frequency artifacts
4. **Use `pre_filt`** to avoid amplifying noise at frequencies where the response is small
5. **Check for polarity reversal** — SmartSolo nodes can have reversed polarity
6. **The 2 Hz geophone extends usable bandwidth** to lower frequencies than the 5 Hz
7. **RESP files describe each stage separately**; PZ files combine everything into one transfer function
8. **Sanity check your results**: ambient noise should be ~0.01–1 µm/s velocity

---

### References & Further Reading

- [ObsPy Documentation](https://docs.obspy.org/)
- [ObsPy Tutorial: Seismometer Correction](https://docs.obspy.org/tutorial/code_snippets/seismometer_correction_simulation.html)
- [AusPass: SmartSolo Nodes](https://www.auspass.edu.au/xwiki/bin/view/Instrumentation/SmartSolo%20Nodes/)
- [AusPass: SmartSolo Polarity Issues](https://auspass.edu.au/xwiki/bin/view/Data/AusPass%20Data/#HSmartSoloNodePolarityIssues)
- [SmartSolo IGU-16HR 3C 5Hz](https://smartsolo.com/igu-16hr-3c.html)
- [SmartSolo IGU-16HR 3C 2Hz](https://smartsolo.com/igu-16hr-3c-2hz-eth.html)
- [EarthScope: SmartSolo](https://epic.earthscope.org/content/instrumentation/all-one-systems/smart-solo)
- [Seismo-Live Notebooks](https://seismo-live.github.io/)
- [IRIS SEED Manual](https://ds.iris.edu/ds/nodes/dmc/data/formats/seed/) — Complete SEED format specification
- [CARLETON Teaching Activity: Instrument Response](https://serc.carleton.edu/teachearth/activities/245398.html)